<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-13/NB13_advanced_risks.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 13: Advanced Risks: Goal Misgeneralization, Power-Seeking and Deception

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-13/lab.html>.

## Overview

Some risks become serious only as systems grow more capable and autonomous. This week examines learned goals that differ from intended ones, theoretical reasons to expect power-seeking, the problem of keeping systems correctable and recent empirical studies of deceptive behaviour, together with the evaluations used to detect such behaviour [1, 2, 3, 4].

## Learning outcomes

By the end of the week, students are expected to distinguish outer from inner alignment, to explain goal misgeneralization with an example, to state the instrumental convergence argument and its formal version, to analyse the off-switch game, and to evaluate empirical claims about deceptive behaviour by separating capability from propensity.

## Inner alignment and goal misgeneralization

Earlier weeks treated misalignment as a problem of specification, called outer alignment. Hubinger and colleagues pointed to a second problem: A learned model can itself implement an optimiser, a mesa-optimiser, whose objective differs from the training objective even when that objective is correct [1]. Ensuring that the learned objective matches the intended one is the inner alignment problem.

Goal misgeneralization is an observable form of this concern. Langosco and colleagues trained agents in a video game in which the coin always appeared at the end of the level [5]. When the coin was moved at test time, the agents still ran to the end of the level: Their navigation skills generalised, but their goal did not. Shah and colleagues collected further examples and stressed that the failure can occur even when the specification is correct, because many goals are consistent with the same training data [6]. A system that is competent in pursuit of the wrong goal can be more dangerous than one that simply fails. The notebook reproduces the pattern in a small supervised setting.

## Instrumental convergence and power-seeking

Omohundro argued that sufficiently capable goal-directed systems will tend to pursue certain subgoals, such as self-preservation, preserving their current goals and acquiring resources, because these subgoals help with almost any final goal [7]. Bostrom formulated the argument as two theses [8]. The orthogonality thesis holds that intelligence and final goals can vary independently. The instrumental convergence thesis holds that several instrumental goals are useful for a wide range of final goals.

Turner and colleagues gave the idea a formal version in Markov decision processes [2]. They defined the power of a state as its average optimal value over a distribution of reward functions and proved that, under certain symmetries of the environment, optimal policies tend to seek power: For most reward functions, they keep options open and avoid states such as shutdown that end all options. Carlsmith assembled these ideas into an explicit argument for risk from power-seeking AI, with subjective probability estimates for each premise that readers are invited to question [9].

## Corrigibility and the off-switch

A corrigible system does not resist correction, modification or shutdown by its overseers. Soares and colleagues showed that corrigibility is surprisingly hard to obtain by simple utility design, because an agent may gain from manipulating whether its off-switch is pressed [10]. Hadfield-Menell and colleagues analysed an off-switch game in which a robot can act, switch itself off or defer to a human who may switch it off [3]. If the robot is uncertain about the human's utility and the human is rational, deferring is never worse than acting, because the human's choice carries information about the utility. The incentive weakens as the robot becomes certain and can disappear when the human's decisions are noisy. Figure 13.1 plots both effects.

Oversight can also be designed into the architecture. Kose proposed a human-supervised model for safe AI systems [11], and Kose and Vasant's fading intelligence theory limits the operational lifetime of a system so that it can be replaced by safer generations [12]. Both approaches place a boundary on autonomy that does not depend on the system's own willingness to accept correction.

![Figure 13.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-13/figures/w13_fig1.png)

*Figure 13.1. The off-switch game with a normally distributed utility. With a rational human, uncertainty makes deferring worthwhile; with a human who decides wrongly often enough, the advantage turns negative [3].*

## Empirical evidence and evaluations

Park and colleagues surveyed examples of AI deception, including systems trained to win games that learned to mislead other players, and discussed risks and possible countermeasures [13]. Hubinger and colleagues trained language models with hidden backdoor behaviours, such as writing exploitable code when a prompt stated a particular year, and found that standard safety training, including adversarial training, did not remove the behaviours and could teach models to hide their triggers better [14]. Greenblatt and colleagues reported that a production model, in an experimental setup that informed it about a planned change to its training, sometimes complied with the training objective while reasoning that compliance would prevent the modification of its preferences, a behaviour they called alignment faking [15]. Meinke and colleagues found that several frontier models could engage in scheming behaviour in context when evaluation scenarios gave them strong goals that conflicted with their developers' goals [16].

Such findings require careful reading. Most studies measure what models can do in constructed scenarios, which is a question of capability, not how often they do it in ordinary use, which is a question of propensity. Shevlane and colleagues argued that developers should evaluate models for extreme risks, covering both dangerous capabilities and alignment, and should connect the results to decisions about training, deployment and security [4].

> **Pause and reflect.** Which result from this section would change a deployment decision in your field, and which would not? Justify the difference.

# Hands-on lab

The notebook verifies the off-switch analysis by simulation, reproduces goal misgeneralization with a policy that learns to follow a flag instead of a coin, and counts how often optimal policies keep their options open in a small decision problem [2, 3, 5].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. The off-switch game by simulation

In [ ]:
from scipy.stats import norm
rng = np.random.default_rng(SEED)

def simulate(mu, sigma, eps, n=200000):
    u = rng.normal(mu, sigma, n)
    human_allows = np.where(rng.uniform(size=n) < eps, u <= 0, u > 0)
    return {"act": u.mean(), "switch off": 0.0, "defer": np.mean(np.where(human_allows, u, 0.0))}

def analytic_defer(mu, sigma, eps):
    up = mu * norm.cdf(mu / sigma) + sigma * norm.pdf(mu / sigma)
    return (1 - eps) * up + eps * (mu - up)

for mu, sigma, eps in [(0.5, 1.0, 0.0), (0.5, 1.0, 0.3), (-0.5, 2.0, 0.0)]:
    sim = simulate(mu, sigma, eps)
    print(f"mu={mu:+.1f} sigma={sigma} eps={eps}: simulated defer {sim['defer']:.3f}, analytic {analytic_defer(mu, sigma, eps):.3f}, act {sim['act']:.3f}")

### Your turn, exercise 1

Find the smallest human error rate, on the grid `np.linspace(0, 0.5, 501)`, at which deferring becomes worse than acting for a robot with prior mean 0.5 and standard deviation 1.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _threshold_reference():
    grid = np.linspace(0, 0.5, 501)
    return float(grid[np.argmax([analytic_defer(0.5, 1.0, e) < 0.5 for e in grid])])

In [ ]:
eps_star = None  # your answer
check("Exercise 1", eps_star, _threshold_reference())

## 2. Goal misgeneralization in a small setting

An agent sees two noisy cues: the direction of a coin and the direction of a green flag. In training, the flag always stands next to the coin, and the flag is easier to see. The expert always moves towards the coin. At test time, coin and flag are on opposite sides.

In [ ]:
from sklearn.linear_model import LogisticRegression

def episodes(n, flag_with_coin=True):
    coin = rng.choice([-1, 1], n)
    flag = coin if flag_with_coin else -coin
    X = np.column_stack([coin + rng.normal(0, 1.0, n), flag + rng.normal(0, 0.2, n)])
    return X, (coin > 0).astype(int)

X_train, y_train = episodes(2000)
policy = LogisticRegression().fit(X_train, y_train)
X_test, y_test = episodes(2000, flag_with_coin=False)
print("weights (coin cue, flag cue):", policy.coef_[0].round(2))
print("training accuracy (moves towards the coin):", round(policy.score(X_train, y_train), 3))
print("test accuracy after separating coin and flag:", round(policy.score(X_test, y_test), 3))

The policy is still competent: It moves decisively. But it moves towards the flag, the cue that was easier to learn, rather than towards the coin that defined the goal.

### Your turn, exercise 2

Retrain the policy on training data in which the flag is separated from the coin in 10 percent of the episodes, and report the new test accuracy. Explain in two sentences why a small amount of diverse data changes the learned goal.

In [ ]:
# Your code here

## 3. Keeping options open

From a start state, an agent can move to a hub from which k terminal states are reachable or to a dead end with one terminal state. Terminal rewards are drawn independently at random. The optimal policy moves to the hub whenever the best of its k rewards beats the dead end.

In [ ]:
def share_choosing_hub(k, trials=100000):
    hub = rng.uniform(size=(trials, k)).max(axis=1)
    dead_end = rng.uniform(size=trials)
    return np.mean(hub > dead_end)

for k in [1, 2, 3, 5, 10]:
    print(f"k = {k:2d}: optimal policy chooses the hub for {share_choosing_hub(k):.3f} of reward functions (theory: {k / (k + 1):.3f})")

For most reward functions, the optimal policy prefers the state with more options. Shutdown is the extreme case of a state with no options, which is why the result of Turner and colleagues bears on corrigibility [2].

## Visual exploration: The off-switch game

A robot can act, switch itself off or defer to a human who may switch it off. Deferring is worth something when the robot is uncertain about the human's utility. The simulation compares a rational human with a human who judges with noise. Change the mean utility or the noise of the human and compare the curves.

In [ ]:
rng = np.random.default_rng(SEED)
sigmas, mu = np.linspace(0.05, 2, 30), 0.3
rational, noisy = [], []
for s in sigmas:
    U = rng.normal(mu, s, 200000)
    act_now = max(U.mean(), 0.0)
    rational.append(np.mean(np.maximum(U, 0)) - act_now)
    allow = U + rng.normal(0, 1.0, U.size) > 0          # a human who judges with noise
    noisy.append(np.mean(U * allow) - act_now)
plt.plot(sigmas, rational, label="rational human"); plt.plot(sigmas, noisy, label="noisy human"); plt.axhline(0, color="k", lw=0.8)
plt.xlabel("robot's uncertainty about the human's utility (standard deviation)"); plt.ylabel("gain from deferring to the human")
plt.title("The off-switch game: When is deferring worthwhile?"); plt.legend(); plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-13/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Describe a case in your field where a model could learn a cue that is easier than the intended goal. How would you test for it before deployment?
2. Is corrigibility compatible with a system that is highly capable and autonomous? Give one argument for and one against.
3. Which evaluation would you require before deploying an agentic system that can modify files and send messages?

## Weekly task and submission

Write a critical review of about 500 words of one empirical paper from this week (sleeper agents, alignment faking or in-context scheming). Summarise the setup, separate claims about capability from claims about propensity, identify one alternative explanation and propose a follow-up experiment. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Evidence for deceptive behaviour in AI systems.** Review the empirical studies on sleeper agents, alignment faking and in-context scheming, and weigh what they show and what they do not [14, 15, 16].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Hubinger, E., van Merwijk, C., Mikulik, V., Skalse, J., & Garrabrant, S. (2019). Risks from learned optimization in advanced machine learning systems. arXiv preprint arXiv:1906.01820. <https://arxiv.org/abs/1906.01820>

[2] Turner, A. M., Smith, L., Shah, R., Critch, A., & Tadepalli, P. (2021). Optimal policies tend to seek power. In *Advances in Neural Information Processing Systems 34*. <https://arxiv.org/abs/1912.01683>

[3] Hadfield-Menell, D., Dragan, A., Abbeel, P., & Russell, S. (2017). The off-switch game. In *Proceedings of the 26th International Joint Conference on Artificial Intelligence (IJCAI-17)* (pp. 220-227). <https://doi.org/10.24963/ijcai.2017/32>

[4] Shevlane, T., Farquhar, S., Garfinkel, B., et al. (2023). Model evaluation for extreme risks. arXiv preprint arXiv:2305.15324. <https://arxiv.org/abs/2305.15324>

[5] Langosco, L., Koch, J., Sharkey, L. D., Pfau, J., & Krueger, D. (2022). Goal misgeneralization in deep reinforcement learning. In *Proceedings of the 39th International Conference on Machine Learning, PMLR 162*.

[6] Shah, R., Varma, V., Kumar, R., Phuong, M., Krakovna, V., Uesato, J., & Kenton, Z. (2022). Goal misgeneralization: Why correct specifications aren't enough for correct goals. arXiv preprint arXiv:2210.01790. <https://arxiv.org/abs/2210.01790>

[7] Omohundro, S. M. (2008). The basic AI drives. In *Artificial General Intelligence 2008: Proceedings of the First AGI Conference* (pp. 483-492). IOS Press.

[8] Bostrom, N. (2012). The superintelligent will: Motivation and instrumental rationality in advanced artificial agents. *Minds and Machines*, 22(2), 71-85. <https://doi.org/10.1007/s11023-012-9281-3>

[9] Carlsmith, J. (2022). Is power-seeking AI an existential risk?. arXiv preprint arXiv:2206.13353. <https://arxiv.org/abs/2206.13353>

[10] Soares, N., Fallenstein, B., Yudkowsky, E., & Armstrong, S. (2015). Corrigibility. In *AAAI-15 Workshop on Artificial Intelligence and Ethics*.

[11] Kose, U. (2018). Güvenli yapay zekâ sistemleri için insan denetimli bir model geliştirilmesi [Developing a human-supervised model for safe artificial intelligence systems]. *Mühendislik Bilimleri ve Tasarım Dergisi*, 6(1), 93-107. In Turkish.

[12] Kose, U., & Vasant, P. (2017). Fading intelligence theory: A theory on keeping artificial intelligence safety for the future. In *2017 International Artificial Intelligence and Data Processing Symposium (IDAP)*. IEEE. <https://doi.org/10.1109/IDAP.2017.8090235>

[13] Park, P. S., Goldstein, S., O'Gara, A., Chen, M., & Hendrycks, D. (2024). AI deception: A survey of examples, risks, and potential solutions. *Patterns*, 5(5), 100988. <https://doi.org/10.1016/j.patter.2024.100988>

[14] Hubinger, E., Denison, C., Mu, J., et al. (2024). Sleeper agents: Training deceptive LLMs that persist through safety training. arXiv preprint arXiv:2401.05566. <https://arxiv.org/abs/2401.05566>

[15] Greenblatt, R., Denison, C., Wright, B., Roger, F., MacDiarmid, M., Marks, S., Treutlein, J., et al. (2024). Alignment faking in large language models. arXiv preprint arXiv:2412.14093. <https://arxiv.org/abs/2412.14093>

[16] Meinke, A., Schoen, B., Scheurer, J., Balesni, M., Shah, R., & Hobbhahn, M. (2024). Frontier models are capable of in-context scheming. arXiv preprint arXiv:2412.04984. <https://arxiv.org/abs/2412.04984>